In [15]:
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
import torch_directml

In [16]:
# Class Dictionary that stores words
class Dictionary:
    def __init__(self):
        self.word2ind = {}
        self.ind2word = {}
        self.idx = 0
    def add_word(self, word):
        if word not in self.word2ind:
            self.word2ind[word] = self.idx
            self.ind2word[self.idx] = word
            self.idx += 1
    def __len__(self):
        return len(self.word2ind)

In [17]:
# Class for processing text
class TextProcessor:
    def __init__(self):
        self.dictionary = Dictionary()
    def get_data(self, path, batch_size=20):
        with open(path, 'r') as f: # Fill the dictionary
            tokens = 0
            for line in f:
                words = line.split() + ['<eos>']
                tokens += len(words)
                for word in words:
                    self.dictionary.add_word(word)
        rep_tensor = torch.LongTensor(tokens) # Get the text representation tensor
        index = 0
        with open(path, 'r') as f:
            for line in f:
                words = line.split() + ['<eos>']
                for word in words:
                    rep_tensor[index] = self.dictionary.word2ind[word]
                    index += 1
        num_batches = tokens // batch_size
        rep_tensor = rep_tensor[:num_batches * batch_size]
        rep_tensor = rep_tensor.view(batch_size, -1)
        return rep_tensor

In [18]:
# Constants
embed_size = 128
hidden_size = 1024
num_layers = 2
num_epochs = 50
batch_size = 20
timesteps = 30
learning_rate = 0.002

In [19]:
# Processor object
corpus = TextProcessor()

In [20]:
# Process the text
rep_tensor = corpus.get_data('./source/RNN/alice.txt', batch_size)

In [21]:
print(rep_tensor.shape)

torch.Size([20, 1484])


In [22]:
vocab_size = len(corpus.dictionary)
print(vocab_size)

5290


In [23]:
num_batches = rep_tensor.shape[1] // timesteps
print(num_batches)

49


In [24]:
# Generator class
class TextGenerator(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.linear = nn.Linear(hidden_size, vocab_size)
    def forward(self, x, h):
        x = self.embed(x)
        out, (h, c) = self.lstm(x, h)
        out = out.reshape(-1, hidden_size)
        out = self.linear(out)
        return out, (h, c)

In [25]:
# Object determination
model = TextGenerator(vocab_size, embed_size, hidden_size, num_layers)
loss_fn = nn.CrossEntropyLoss(reduction='mean')
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

In [26]:
# Device
device = torch_directml.device(0)
print(torch_directml.device_name(0))

AMD Radeon RX 6800S 


In [27]:
# Training
model.train()
for epoch in range(num_epochs):
    states = (torch.zeros(num_layers, batch_size, hidden_size),
              torch.zeros(num_layers, batch_size, hidden_size))
    for i in range(0, rep_tensor.size(1) - timesteps, timesteps):
        inputs = rep_tensor[:, i : i + timesteps]
        targets = rep_tensor[:, (i + 1) : (i + 1) + timesteps]
        outputs, states = model(inputs, states)
        states = (states[0].detach(), states[1].detach())
        loss = loss_fn(outputs, targets.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

        step = (i + 1) // timesteps
        if step % 100 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")
    scheduler.step()

Epoch 1/50, Loss: 8.5763
Epoch 2/50, Loss: 6.7488
Epoch 3/50, Loss: 6.0602
Epoch 4/50, Loss: 5.7693
Epoch 5/50, Loss: 5.5436
Epoch 6/50, Loss: 5.4087
Epoch 7/50, Loss: 5.4385
Epoch 8/50, Loss: 5.2222
Epoch 9/50, Loss: 4.9093
Epoch 10/50, Loss: 4.7825
Epoch 11/50, Loss: 4.5618
Epoch 12/50, Loss: 4.6827
Epoch 13/50, Loss: 4.4408
Epoch 14/50, Loss: 4.1261
Epoch 15/50, Loss: 4.0085
Epoch 16/50, Loss: 3.8929
Epoch 17/50, Loss: 3.7446
Epoch 18/50, Loss: 3.5464
Epoch 19/50, Loss: 3.3109
Epoch 20/50, Loss: 3.4089
Epoch 21/50, Loss: 3.1550
Epoch 22/50, Loss: 3.0083
Epoch 23/50, Loss: 2.7307
Epoch 24/50, Loss: 2.6254
Epoch 25/50, Loss: 2.4646
Epoch 26/50, Loss: 2.2476
Epoch 27/50, Loss: 2.1266
Epoch 28/50, Loss: 1.9807
Epoch 29/50, Loss: 1.8079
Epoch 30/50, Loss: 1.6776
Epoch 31/50, Loss: 1.4956
Epoch 32/50, Loss: 1.4541
Epoch 33/50, Loss: 1.3408
Epoch 34/50, Loss: 1.2559
Epoch 35/50, Loss: 1.1519
Epoch 36/50, Loss: 1.0747
Epoch 37/50, Loss: 0.9914
Epoch 38/50, Loss: 0.8972
Epoch 39/50, Loss: 0.

In [28]:
# Genenrating text
model.eval()
with torch.no_grad():
    with open('./source/RNN/result.txt', 'w') as f:
        states = (torch.zeros(num_layers, 1, hidden_size),
              torch.zeros(num_layers, 1, hidden_size))
        inp = torch.randint(0, vocab_size, (1,)).long().unsqueeze(1)
        for i in range(500):
            outputs, _ = model(inp, states)
            print(outputs.shape)
            prob = outputs.exp()
            # Takes a random word from distribution with its probability
            word_id = torch.multinomial(prob, num_samples=1).item()
            print(word_id)
            inp.fill_(word_id)

            word = corpus.dictionary.ind2word[word_id]
            word = '\n' if word == '<eos>' else word + ' '
            f.write(word)
            if (i + 1) % 100 == 0:
                print(f"Sampled {i+1}/{500} words and save to {'./source/RNN/result.txt'}")

torch.Size([1, 5290])
2479
torch.Size([1, 5290])
1060
torch.Size([1, 5290])
25
torch.Size([1, 5290])
224
torch.Size([1, 5290])
1042
torch.Size([1, 5290])
7
torch.Size([1, 5290])
15
torch.Size([1, 5290])
3
torch.Size([1, 5290])
3779
torch.Size([1, 5290])
4277
torch.Size([1, 5290])
5
torch.Size([1, 5290])
5
torch.Size([1, 5290])
5
torch.Size([1, 5290])
3775
torch.Size([1, 5290])
9
torch.Size([1, 5290])
4239
torch.Size([1, 5290])
1644
torch.Size([1, 5290])
5
torch.Size([1, 5290])
5
torch.Size([1, 5290])
5
torch.Size([1, 5290])
5
torch.Size([1, 5290])
5
torch.Size([1, 5290])
5
torch.Size([1, 5290])
1
torch.Size([1, 5290])
3060
torch.Size([1, 5290])
97
torch.Size([1, 5290])
44
torch.Size([1, 5290])
86
torch.Size([1, 5290])
7
torch.Size([1, 5290])
575
torch.Size([1, 5290])
1208
torch.Size([1, 5290])
4065
torch.Size([1, 5290])
1665
torch.Size([1, 5290])
5
torch.Size([1, 5290])
668
torch.Size([1, 5290])
3035
torch.Size([1, 5290])
2783
torch.Size([1, 5290])
3594
torch.Size([1, 5290])
796
torch.